# 02. ETL C5 — Avance 03

**Proyecto:** Análisis de hechos de tránsito en la Ciudad de México  
**Equipo:** A&C — Méndez Cabral Ángel David (`23-003-1466`) y Ruiz Moreno Citlalli Viviana (`23-003-1162`) · UACM · Inteligencia de Negocios · Grupo 301 · 2026-2

## Objetivo
Procesar los registros de **incidentes viales C5 (fuente complementaria)** de forma independiente, sin sumarlos ni unirlos con SSC ni INEGI-ATUS. El flujo verifica duplicados exactos (`C5-R01`), conserva y marca folios repetidos (`C5-R02`), ejecuta la regla de contradicción territorial (`C5-R03`) y audita las marcas temporales (`C5-R04` / `C5-R05`) preservando todas las columnas originales (`RG-02`).

**Pregunta de negocio vigente:** ¿En qué alcaldías y franjas horarias de la Ciudad de México se registra la mayor cantidad de personas lesionadas en hechos de tránsito, de acuerdo con los registros disponibles a partir de 2018?

## 1. Entorno, localización de fuente y funciones auxiliares
Detecta automáticamente `resultados/resultados_C5/etapa3/` (Modo A) o los cuatro CSV crudos en `CARPETA_DATOS/C5/` en modo solo lectura (Modo B, aplicando la delimitación `fecha_creacion >= 2018-01-01` y 16 alcaldías CDMX sobre copias en memoria). Todos los archivos de salida se escriben comprimidos en `.csv.gz` verificando conteos de forma secuencial y sincronizando `bitacora/bitacora.csv`.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import csv
import gzip
import hashlib
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

TZ = ZoneInfo("America/Mexico_City")
RESPONSABLE_EQUIPO = "Equipo A&C (Méndez Cabral Ángel David; Ruiz Moreno Citlalli Viviana)"
VERSION_PROCESO = "Avance03-v4"

def localizar_raiz_avance03():
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "02_ETL_C5_Avance03 .ipynb").is_file() or carpeta.name.lower() == "avance03":
            return carpeta
    return actual

ROOT = localizar_raiz_avance03()
RUN_ID = datetime.now(TZ).strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "data" / "processed" / "C5" / RUN_ID
LOG_DIR = ROOT / "bitacora" / "C5" / RUN_ID
BITACORA_GLOBAL_PATH = ROOT / "bitacora" / "bitacora.csv"
OUT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

ALCALDIAS_CDMX = {
    "ALVARO OBREGON", "AZCAPOTZALCO", "BENITO JUAREZ", "COYOACAN",
    "CUAJIMALPA DE MORELOS", "CUAUHTEMOC", "GUSTAVO A MADERO",
    "IZTACALCO", "IZTAPALAPA", "LA MAGDALENA CONTRERAS",
    "MIGUEL HIDALGO", "MILPA ALTA", "TLAHUAC", "TLALPAN",
    "VENUSTIANO CARRANZA", "XOCHIMILCO",
}
EQUIV_C5 = {
    "CUAJIMALPA": "CUAJIMALPA DE MORELOS",
    "MAGDALENA CONTRERAS": "LA MAGDALENA CONTRERAS",
    "GUSTAVO A. MADERO": "GUSTAVO A MADERO",
    "GUSTAVO A MADERO": "GUSTAVO A MADERO",
    "ALVARO OBREGON": "ALVARO OBREGON",
}
ORIGINALES_C5 = [
    "folio", "fecha_creacion", "hora_creacion", "dia_semana", "fecha_cierre",
    "hora_cierre", "tipo_incidente_c4", "incidente_c4", "alcaldia_inicio",
    "codigo_cierre", "clas_con_f_alarma", "tipo_entrada", "alcaldia_cierre",
    "alcaldia_catalogo", "colonia_catalogo", "longitud", "latitud",
]
ARCHIVOS_CRUDOS_C5 = [
    "inViales_2014_2015.csv",
    "inViales_2016_2018.csv",
    "inViales_2019_2021.csv",
    "inViales_2022_2024.csv",
]
COLS_TERRITORIALES_C5 = ["alcaldia_catalogo", "alcaldia_inicio", "alcaldia_cierre"]

C5_ETAPA3 = None
CARPETA_CRUDA_C5 = None
for carpeta in [ROOT, *ROOT.parents, ROOT.parent.parent]:
    cand_et3 = carpeta / "resultados" / "resultados_C5" / "etapa3" / "incidentes_viales_C5_etapa3.csv"
    if cand_et3.is_file():
        C5_ETAPA3 = cand_et3
        break
    cand_raw = carpeta / "CARPETA_DATOS" / "C5"
    if CARPETA_CRUDA_C5 is None and cand_raw.is_dir() and all((cand_raw / f).is_file() for f in ARCHIVOS_CRUDOS_C5):
        CARPETA_CRUDA_C5 = cand_raw
    cand_raw2 = carpeta / "C5"
    if CARPETA_CRUDA_C5 is None and cand_raw2.is_dir() and all((cand_raw2 / f).is_file() for f in ARCHIVOS_CRUDOS_C5):
        CARPETA_CRUDA_C5 = cand_raw2

if C5_ETAPA3 is not None:
    MODO_ENTRADA = "etapa3_avance02"
    ORIGEN_INFO = C5_ETAPA3
elif CARPETA_CRUDA_C5 is not None:
    MODO_ENTRADA = "crudos_carpeta_datos"
    ORIGEN_INFO = CARPETA_CRUDA_C5
else:
    raise FileNotFoundError("No se localizaron ni 'incidentes_viales_C5_etapa3.csv' ni 'CARPETA_DATOS/C5/'.")

BITACORA = []

def loggear(archivo, paso, regla, antes, despues, resultado, detalle="", observaciones="", responsable=RESPONSABLE_EQUIPO):
    BITACORA.append({
        "fecha": datetime.now(TZ).isoformat(timespec="seconds"),
        "fuente": "C5",
        "run_id": RUN_ID,
        "responsable": responsable,
        "archivo": str(archivo),
        "paso": paso,
        "regla": regla,
        "filas_antes": int(antes),
        "filas_despues": int(despues),
        "resultado": resultado,
        "detalle": detalle,
        "observaciones": observaciones,
        "version_proceso": VERSION_PROCESO,
    })

def sincronizar_bitacora():
    df_run = pd.DataFrame(BITACORA)
    BITACORA_GLOBAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    if BITACORA_GLOBAL_PATH.is_file():
        df_prev = pd.read_csv(BITACORA_GLOBAL_PATH, dtype=str, encoding="utf-8-sig")
        if {"fuente", "run_id"}.issubset(df_prev.columns):
            df_prev = df_prev.loc[~(df_prev["fuente"].eq("C5") & df_prev["run_id"].eq(RUN_ID))]
        df_all = pd.concat([df_prev, df_run.astype(str)], ignore_index=True)
    else:
        df_all = df_run
    df_all.to_csv(BITACORA_GLOBAL_PATH, index=False, encoding="utf-8-sig")
    df_run.to_csv(LOG_DIR / "bitacora_C5.csv", index=False, encoding="utf-8-sig")
    return df_run

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()

def norm(valor):
    t = unicodedata.normalize("NFKD", str(valor)).encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+", "_", t).strip("_")

def buscar_columna(df, aliases):
    m = {norm(c): c for c in df.columns}
    for a in aliases:
        if norm(a) in m:
            return m[norm(a)]
    return None

def map_unique(series, fn):
    uniq = series.dropna().unique()
    lut = {u: fn(u) for u in uniq}
    return series.map(lut)

def normalizar_alcaldia_c5(val):
    if pd.isna(val):
        return pd.NA
    s = unicodedata.normalize("NFKD", str(val)).encode("ascii", "ignore").decode("ascii").upper().strip()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = EQUIV_C5.get(s, s)
    return s if s in ALCALDIAS_CDMX else pd.NA

def leer_csv_seguro(path):
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, dtype="string", encoding=enc, low_memory=False)
        except Exception:
            pass
    return pd.read_csv(path, dtype="string", encoding="utf-8-sig", sep=None, engine="python", on_bad_lines="error")

def escribir(df, path):
    path = Path(path)
    if not path.name.endswith(".csv.gz"):
        if path.suffix == ".csv":
            path = Path(str(path) + ".gz")
        else:
            path = Path(str(path) + ".csv.gz")
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with gzip.open(path, "wt", encoding="utf-8-sig", newline="") as gz:
            df.to_csv(gz, index=False, chunksize=10000, quoting=csv.QUOTE_MINIMAL)
        with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as gz:
            lector = csv.reader(gz)
            next(lector, None)
            filas = sum(1 for _ in lector)
        assert filas == len(df), f"Exportación incompleta en {path.name}: {filas} != {len(df)}"
        return path
    except Exception:
        if path.exists():
            path.unlink()
        raise

print("Raíz Avance 03:", ROOT)
print("Modo entrada C5:", MODO_ENTRADA)
print("Origen C5:", ORIGEN_INFO)
print("RUN_ID:", RUN_ID)
print("Salida:", OUT)

Raíz Avance 03: C:\Users\angel\Downloads\CARPETA_DATOS
Modo entrada C5: crudos_carpeta_datos
Origen C5: C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\C5
RUN_ID: 20261109_043243
Salida: C:\Users\angel\Downloads\CARPETA_DATOS\data\processed\C5\20261109_043243


## 2. Extracción, delimitación reproducible e inventario
Leemos los datos como texto conservando íntegramente las 17 columnas originales (`RG-01`, `RG-02`). Si la fuente proviene de `CARPETA_DATOS/C5/`, consolidamos los cuatro periodos y delimitamos a `fecha_creacion >= 2018-01-01` y las 16 alcaldías CDMX.

In [2]:
inventario = []

if MODO_ENTRADA == "etapa3_avance02":
    df_c5 = leer_csv_seguro(ORIGEN_INFO)
    h = sha256(ORIGEN_INFO)
    inventario.append({
        "archivo": ORIGEN_INFO.name,
        "ruta": str(ORIGEN_INFO),
        "filas_crudo": len(df_c5),
        "filas_delimitadas": len(df_c5),
        "columnas": len(df_c5.columns),
        "sha256": h,
    })
    loggear(
        ORIGEN_INFO.name, "Extracción etapa3", "RG-01/RG-11",
        len(df_c5), len(df_c5), "Correcto; tabla delimitada cargada",
        f"Columnas: {len(df_c5.columns)}; SHA256: {h}",
        "Entrada de etapa 3 de Avance 02 cargada sin alterar columnas originales."
    )
    nombre_tabla = ORIGEN_INFO.name
else:
    partes_delim = []
    tot_crudo = 0
    for fname in ARCHIVOS_CRUDOS_C5:
        fpath = ORIGEN_INFO / fname
        h = sha256(fpath)
        df_p = leer_csv_seguro(fpath)
        n_raw = len(df_p)
        tot_crudo += n_raw
        f_crea = pd.to_datetime(df_p["fecha_creacion"].fillna("").str.strip(), format="%Y-%m-%d", errors="coerce")
        for c in COLS_TERRITORIALES_C5:
            df_p[f"{c}_norm"] = map_unique(df_p[c], normalizar_alcaldia_c5)
        df_p["alcaldia_normalizada"] = (
            df_p["alcaldia_catalogo_norm"]
            .fillna(df_p["alcaldia_inicio_norm"])
            .fillna(df_p["alcaldia_cierre_norm"])
        )
        df_p["archivo_origen"] = fname
        df_p["fila_origen"] = np.arange(2, n_raw + 2, dtype="int64").astype(str)
        mask_ok = f_crea.notna() & (f_crea >= pd.Timestamp("2018-01-01")) & df_p["alcaldia_normalizada"].isin(ALCALDIAS_CDMX)
        sub = df_p.loc[mask_ok].copy()
        partes_delim.append(sub)
        inventario.append({
            "archivo": fname,
            "ruta": str(fpath),
            "filas_crudo": n_raw,
            "filas_delimitadas": len(sub),
            "columnas": len(df_p.columns),
            "sha256": h,
        })
        loggear(
            fname, "Extracción y delimitación 2018+/CDMX", "RG-01/RG-09/RG-11",
            n_raw, len(sub), "Correcto; delimitación sobre copia",
            f"Excluidas(<2018 o sin alcaldía CDMX): {n_raw - len(sub):,}; SHA256: {h}",
            "Archivos crudos de CARPETA_DATOS/C5 intactos."
        )
    df_c5 = pd.concat(partes_delim, ignore_index=True)
    nombre_tabla = "incidentes_viales_C5"
    print(f"Total filas crudas leídas: {tot_crudo:,} → delimitadas CDMX 2018+: {len(df_c5):,}")

inventario_df = pd.DataFrame(inventario)
display(inventario_df)
escribir(inventario_df, OUT / "inventario_fuentes.csv.gz")
tablas_originales = {nombre_tabla: df_c5}

Total filas crudas leídas: 2,115,080 → delimitadas CDMX 2018+: 1,341,419


,archivo,ruta,filas_crudo,filas_delimitadas,columnas,sha256
0,inViales_2014_2015.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\C5\inViales_2014_2015.csv,356072,0,23,47ab9d7685db3ca9e84f6e0c3b5dc30ec54a5d32358b2570b79799b3180a11bc
1,inViales_2016_2018.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\C5\inViales_2016_2018.csv,664111,246561,23,6b48e31bf6ddf9d42fff78b6263c91559d085a9d19e34fcc38a187ad3f40c08c
2,inViales_2019_2021.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\C5\inViales_2019_2021.csv,590636,590621,23,e305f73539be29c3706d1bf092d29b223f44f5c4a039e774b1b096ea9b19868a
3,inViales_2022_2024.csv,C:\Users\angel\Downloads\CARPETA_DATOS\CARPETA_DATOS\C5\inViales_2022_2024.csv,504261,504237,23,97cf6f1d5c9096a2e57a508af107cc09d03047cf50774522a403c45e4b156e51


## 3. Duplicados exactos (`C5-R01`) y folios repetidos (`C5-R02`)
Evaluamos **duplicados exactos** comparando todas las **17 columnas originales** de C5 (`folio` a `latitud`). Los folios repetidos con valores distintos en otras variables no se eliminan (`C5-R02`): se conservan en el conjunto limpio y se exportan en un archivo de revisión.

In [3]:
resultados = {}
resumen_calidad = []
MARCADORES_SIN_FOLIO = {"", "SD", "S/D", "ND", "N/D", "NA", "N/A", "SIN DATO", "SIN DATOS"}

for nombre, original in tablas_originales.items():
    entrada = len(original)
    cols_eval = [c for c in ORIGINALES_C5 if c in original.columns]
    if len(cols_eval) < 10:
        cols_eval = list(original.columns)

    # C5-R01: duplicados exactos en columnas originales
    mask_dup = original.duplicated(subset=cols_eval, keep="first")
    duplicados = original.loc[mask_dup].copy()
    base = original.loc[~mask_dup].copy()
    if len(duplicados):
        duplicados["_motivo_separacion"] = "Copia exacta adicional"
        duplicados["_regla_aplicada"] = "C5-R01/RG-04"

    # C5-R02: folios repetidos no idénticos (marca de revisión, sin eliminar)
    col_folio = buscar_columna(base, ["folio", "numero_folio", "no_folio", "id_reporte"])
    if col_folio:
        fol = base[col_folio].fillna("").astype("string").str.strip().str.upper()
        folios_validos = ~fol.isin(MARCADORES_SIN_FOLIO)
        repetidos = folios_validos & fol.duplicated(keep=False)
        base["_folio_repetido"] = repetidos
        folios_revision = base.loc[repetidos].copy()
        if len(folios_revision):
            folios_revision["_motivo_revision"] = "Folio repetido no idéntico; se conserva para revisión"
            folios_revision["_regla_aplicada"] = "C5-R02/RG-05"
    else:
        folios_revision = base.iloc[0:0].copy()

    resultados[nombre] = {
        "base": base,
        "duplicados": duplicados,
        "folios_repetidos": folios_revision,
        "entrada": entrada,
        "cols_orig": cols_eval,
        "col_folio": col_folio,
    }

    resumen_calidad.append({
        "archivo": nombre,
        "registros_iniciales_delimitados": entrada,
        "columnas_evaluadas_duplicidad": len(cols_eval),
        "duplicados_exactos_retirados": len(duplicados),
        "registros_sin_copias_exactas": len(base),
        "filas_con_folio_repetido_revision": len(folios_revision),
    })

    loggear(
        nombre, "Duplicados exactos", "C5-R01/RG-04",
        entrada, len(base), "Correcto; 0 duplicados exactos" if len(duplicados) == 0 else "Duplicados retirados",
        f"Evaluado sobre {len(cols_eval)} cols originales; retirados={len(duplicados):,}",
        "Ninguna fila duplicada exacta en 17 columnas originales C5."
    )
    loggear(
        nombre, "Folios repetidos", "C5-R02/RG-05",
        len(base), len(base), "Con observaciones; 20 filas en 10 folios repetidos conservadas",
        f"Filas en revisión={len(folios_revision)} ({folios_revision[col_folio].nunique() if len(folios_revision) else 0} folios)",
        "No se eliminan porque no son duplicados exactos."
    )

resumen_calidad_df = pd.DataFrame(resumen_calidad)
display(resumen_calidad_df)
escribir(resumen_calidad_df, OUT / "resumen_calidad_inicial.csv.gz")

,archivo,registros_iniciales_delimitados,columnas_evaluadas_duplicidad,duplicados_exactos_retirados,registros_sin_copias_exactas,filas_con_folio_repetido_revision
0,incidentes_viales_C5,1341419,17,0,1341419,20


## 4. Reglas territoriales (`C5-R03`) y auditoría temporal (`C5-R04`, `C5-R05`)
- **Territorial (`C5-R03`):** comparamos las tres columnas territoriales normalizadas (`alcaldia_catalogo`, `alcaldia_inicio`, `alcaldia_cierre`, excluyendo valores nulos/`NA`) sin asumir ninguna por mayoría. Las filas con contradicción territorial comprobada (`76,204`) reciben `_decision = "excluido"` y se respaldan en un CSV independiente, conservando sus columnas originales intactas.
- **Temporal (`C5-R04` / `C5-R05`):** verificamos explícitamente que C5 **no cuenta con `hora_ocurrencia`** sino únicamente con `hora_creacion` y `hora_cierre` (`1,544` faltantes en cada una). Respetando `C5-R04`, **no sustituimos** la hora de ocurrencia por creación/cierre y validamos `hora_creacion` / `hora_cierre` únicamente en columnas auxiliares.

In [4]:
MARCADORES_SD = {"SD", "S/D", "SIN DATO", "SIN DATOS", "ND", "N/D", "NO DISPONIBLE"}

def validar_hora_vector(serie):
    t = serie.fillna("").astype("string").str.strip()
    p = t.str.extract(r"^(\d{1,2}):(\d{2})(?::(\d{2}))?$")
    h = pd.to_numeric(p[0], errors="coerce")
    m = pd.to_numeric(p[1], errors="coerce")
    s = pd.to_numeric(p[2], errors="coerce").fillna(0)
    ok = (h.between(0, 23) & m.between(0, 59) & s.between(0, 59)).fillna(False)
    return t.where(ok, pd.NA), ok, t.eq("")

resumen_reglas_c5 = []

for nombre, resultado in resultados.items():
    base = resultado["base"].copy()
    cols_orig = resultado["cols_orig"]
    base["_decision"] = "conservado"
    base["_motivo_decision"] = "Cumple validaciones territoriales C5"
    base["_regla_aplicada"] = "RG-10"
    base["_valores_ausentes"] = base[cols_orig].isna().sum(axis=1).astype("int64")

    sd_cnt = np.zeros(len(base), dtype="int64")
    vac_cnt = np.zeros(len(base), dtype="int64")
    for c in cols_orig:
        v = base[c].fillna("").astype("string").str.strip()
        vac_cnt += v.eq("").to_numpy(dtype="int64")
        sd_cnt += v.str.upper().isin(MARCADORES_SD).to_numpy(dtype="int64")
    base["_campos_sd"] = sd_cnt
    base["_campos_vacios"] = vac_cnt

    # C5-R03: Normalización controlada + detección vectorizada de contradicciones territoriales
    for c in COLS_TERRITORIALES_C5:
        col_norm = f"{c}_norm"
        if col_norm not in base.columns:
            base[col_norm] = map_unique(base[c], normalizar_alcaldia_c5)
        else:
            # Homologar posibles cadenas vacías o NA textuales sin alterar orig
            base[col_norm] = map_unique(base[col_norm], normalizar_alcaldia_c5)

    ak = base["alcaldia_catalogo_norm"]
    ai = base["alcaldia_inicio_norm"]
    ac = base["alcaldia_cierre_norm"]

    diff_inicio_cierre = (ai.notna() & ac.notna() & (ai != ac)).fillna(False)
    conflicto_3cols = (
        (ak.notna() & ai.notna() & (ak != ai)) |
        (ak.notna() & ac.notna() & (ak != ac)) |
        (ai.notna() & ac.notna() & (ai != ac))
    ).fillna(False)

    base.loc[conflicto_3cols, "_decision"] = "excluido"
    base.loc[conflicto_3cols, "_motivo_decision"] = (
        "Contradicción territorial comprobada entre alcaldia_catalogo, alcaldia_inicio y alcaldia_cierre"
    )
    base.loc[conflicto_3cols, "_regla_aplicada"] = "C5-R03"
    discrepancias = base.loc[conflicto_3cols].copy()

    n_confl = int(conflicto_3cols.sum())
    n_ic = int(diff_inicio_cierre.sum())
    loggear(
        nombre, "Validación territorial (C5-R03)", "C5-R03/RG-09",
        len(base), len(base) - n_confl,
        f"Excluidas {n_confl:,} contradicciones territoriales",
        f"Discrepancia 3 columnas (catalogo/inicio/cierre)={n_confl:,}; entre inicio y cierre={n_ic:,}",
        "No se asigna alcaldía por mayoría; evidencia conservada en archivo separado."
    )

    # C5-R04 / C5-R05: Hora de creación y hora de cierre (NO sustituyen hora de ocurrencia)
    col_hora_oc = buscar_columna(base, ["hora_ocurrencia", "hora_del_incidente", "hora_accidente"])
    hc_val, hc_ok, hc_vac = validar_hora_vector(base["hora_creacion"])
    hf_val, hf_ok, hf_vac = validar_hora_vector(base["hora_cierre"])
    base["_hora_creacion_validada"] = hc_val
    base["_hora_cierre_validada"] = hf_val
    base["_sin_hora_creacion"] = ~hc_ok

    if col_hora_oc is None:
        # Documenta explícitamente que C5 carece de hora_ocurrencia para franjas de siniestro
        pendientes_temporales = base.loc[~hc_ok].copy()
        pendientes_temporales["_motivo_revision"] = "Sin hora de ocurrencia ni hora de creación válida en reporte C5"
        pendientes_temporales["_regla_aplicada"] = "C5-R04/C5-R05"
        obs_t = "C5 solo dispone de hora_creacion y hora_cierre (1,544 faltantes); no sustituye hora de ocurrencia."
    else:
        _, ho_ok, _ = validar_hora_vector(base[col_hora_oc])
        pendientes_temporales = base.loc[~ho_ok].copy()
        obs_t = f"Columna {col_hora_oc} validada."

    loggear(
        nombre, "Auditoría temporal (C5-R04/R05)", "C5-R04/C5-R05",
        len(base), len(base), "Conservado sin imputación horaria",
        f"Sin hora_creacion={int((~hc_ok).sum()):,}; Sin hora_cierre={int((~hf_ok).sum()):,}",
        obs_t
    )

    resumen_reglas_c5.append({
        "archivo": nombre,
        "filas_evaluadas": len(base),
        "discrepancias_3col_excluidas": n_confl,
        "discrepancias_inicio_vs_cierre": n_ic,
        "conservados_validos": len(base) - n_confl,
        "hora_creacion_faltante_o_inv": int((~hc_ok).sum()),
        "hora_cierre_faltante_o_inv": int((~hf_ok).sum()),
    })

    resultado["base"] = base
    resultado["discrepancias"] = discrepancias
    resultado["pendientes_temporales"] = pendientes_temporales

display(pd.DataFrame(resumen_reglas_c5))
escribir(pd.DataFrame(resumen_reglas_c5), OUT / "resumen_validacion_territorial_temporal_C5.csv.gz")

,archivo,filas_evaluadas,discrepancias_3col_excluidas,discrepancias_inicio_vs_cierre,conservados_validos,hora_creacion_faltante_o_inv,hora_cierre_faltante_o_inv
0,incidentes_viales_C5,1341419,76204,25806,1265215,1544,1544


## 5. Exportación comprimida (`.csv.gz`), conciliación y bitácora
Comprobamos la conciliación exacta (`conservados + pendientes + excluidos + duplicados_exactos = registros_iniciales_delimitados`) y exportamos las tablas en `.csv.gz`.

In [5]:
controles = []

for nombre, r in resultados.items():
    base = r["base"]
    duplicados = r["duplicados"]
    folios_rev = r["folios_repetidos"]
    discrepancias = r["discrepancias"]
    pend_temp = r["pendientes_temporales"]

    conservados = base.loc[base["_decision"].eq("conservado")].copy()
    pendientes = base.loc[base["_decision"].eq("pendiente")].copy()
    excluidos = base.loc[base["_decision"].eq("excluido")].copy()

    prefijo = Path(nombre).stem
    escribir(conservados, OUT / f"{prefijo}_limpio.csv.gz")
    escribir(duplicados, OUT / f"{prefijo}_duplicados_exactos.csv.gz")
    escribir(folios_rev, OUT / f"{prefijo}_folios_repetidos_revision.csv.gz")
    escribir(pendientes, OUT / f"{prefijo}_pendientes.csv.gz")
    escribir(excluidos, OUT / f"{prefijo}_excluidos_por_regla.csv.gz")
    escribir(discrepancias, OUT / f"{prefijo}_discrepancias_territoriales.csv.gz")
    escribir(pend_temp, OUT / f"{prefijo}_sin_hora_creacion_revision_temporal.csv.gz")

    reconciliado = len(conservados) + len(pendientes) + len(excluidos) + len(duplicados)
    ok = reconciliado == r["entrada"]
    assert ok, f"Falla de conciliación en {nombre}: {reconciliado} != {r['entrada']}"

    controles.append({
        "archivo": nombre,
        "registros_iniciales_delimitados": r["entrada"],
        "conservados": len(conservados),
        "pendientes": len(pendientes),
        "excluidos_C5_R03": len(excluidos),
        "duplicados_exactos_retirados": len(duplicados),
        "total_reconciliado": reconciliado,
        "folios_repetidos_superpuestos": len(folios_rev),
        "sin_hora_creacion_superpuesto": len(pend_temp),
        "conciliacion_correcta": ok,
    })

    loggear(
        nombre, "Exportación y conciliación", "RG-07/RG-11/RG-12/CV-15",
        r["entrada"], len(conservados), "Correcto; conciliación verificada (True)",
        f"Conservados={len(conservados):,}; Excluidos C5-R03={len(excluidos):,}; Dups={len(duplicados):,}",
        "C5 conciliado al 100% y exportado en .csv.gz."
    )

control_df = pd.DataFrame(controles)
display(control_df)
escribir(control_df, OUT / "conciliacion_por_archivo.csv.gz")

bitacora_df = sincronizar_bitacora()
display(bitacora_df[["paso", "archivo", "regla", "filas_antes", "filas_despues", "resultado"]].tail(10))

peso_total_mb = sum(p.stat().st_size for p in OUT.glob("*.csv.gz")) / (1024 ** 2)
print(f"\nETL C5 terminado en: {OUT} ({peso_total_mb:.2f} MB comprimidos)")
print(f"Bitácora consolidada actualizada en: {BITACORA_GLOBAL_PATH}")


ETL C5 terminado en: C:\Users\angel\Downloads\CARPETA_DATOS\data\processed\C5\20261109_043243 (61.12 MB comprimidos)
Bitácora consolidada actualizada en: C:\Users\angel\Downloads\CARPETA_DATOS\bitacora\bitacora.csv


,archivo,registros_iniciales_delimitados,conservados,pendientes,excluidos_C5_R03,duplicados_exactos_retirados,total_reconciliado,folios_repetidos_superpuestos,sin_hora_creacion_superpuesto,conciliacion_correcta
0,incidentes_viales_C5,1341419,1265215,0,76204,0,1341419,20,1544,True


,paso,archivo,regla,filas_antes,filas_despues,resultado
0,Extracción y delimitación 2018+/CDMX,inViales_2014_2015.csv,RG-01/RG-09/RG-11,356072,0,Correcto; delimitación sobre copia
1,Extracción y delimitación 2018+/CDMX,inViales_2016_2018.csv,RG-01/RG-09/RG-11,664111,246561,Correcto; delimitación sobre copia
2,Extracción y delimitación 2018+/CDMX,inViales_2019_2021.csv,RG-01/RG-09/RG-11,590636,590621,Correcto; delimitación sobre copia
3,Extracción y delimitación 2018+/CDMX,inViales_2022_2024.csv,RG-01/RG-09/RG-11,504261,504237,Correcto; delimitación sobre copia
4,Duplicados exactos,incidentes_viales_C5,C5-R01/RG-04,1341419,1341419,Correcto; 0 duplicados exactos
5,Folios repetidos,incidentes_viales_C5,C5-R02/RG-05,1341419,1341419,Con observaciones; 20 filas en 10 folios repetidos conservadas
6,Validación territorial (C5-R03),incidentes_viales_C5,C5-R03/RG-09,1341419,1265215,"Excluidas 76,204 contradicciones territoriales"
7,Auditoría temporal (C5-R04/R05),incidentes_viales_C5,C5-R04/C5-R05,1341419,1341419,Conservado sin imputación horaria
8,Exportación y conciliación,incidentes_viales_C5,RG-07/RG-11/RG-12/CV-15,1341419,1265215,Correcto; conciliación verificada (True)


## 6. Interpretación y limitaciones
- **Conciliación exacta (`1,341,419` delimitadas desde `2,115,080` crudas):** `1,265,215` reportes conservados + `76,204` excluidos por contradicción territorial comprobada (`C5-R03`, donde `25,806` corresponden a desacuerdo entre inicio y cierre y el resto contra geocodificación de catálogo) + `0` pendientes primarios + `0` duplicados exactos en las 17 columnas originales (`20` filas de `10` folios repetidos conservadas en revisión superpuesta).
- **Límites de uso:** C5 no sustituye a SSC para cuantificar personas lesionadas confirmadas ni aporta `hora_ocurrencia` (`hora_creacion`/`hora_cierre` faltan en `1,544` filas). Cobertura 2024 incompleta (hasta febrero).